# Tuva Demo Validation

Runs against a local DuckDB build (`tuva_demo.duckdb`, sitting next to this notebook) rather than Databricks, so it's fast and free to re-run. The DB already has the full project built (claims_preprocessing/core/readmissions/ccsr/provider_attribution/ahrq_measures) plus the DQI data-quality layer (`enable_input_layer_testing`) layered on top.

Sections:
1. Build & test summary (from `run_results.json`, captured from the full base build)
2. DQI input-layer data quality summary (`data_quality__data_quality_for_pbi`)
3. Business-logic sanity checks on the active marts (core, readmissions, ccsr, provider attribution, ahrq_measures pqi)
4. Databricks cross-check (static reference numbers from the last verified Databricks run, for comparison)

**To refresh the data this notebook reads**, from the `demo` project root (a full rebuild is slow -- 1.5-3+ hours -- and this machine's 15GB RAM makes even single-threaded runs OOM-risky over that long a session, so prefer the scoped form when you only need to refresh one part):

```
source ~/dbt-env/bin/activate

# full rebuild (rarely needed -- only if claims_preprocessing/core/etc. logic changed)
dbt build --target duckdb_local --vars '{enable_input_layer_testing: true}' --threads 1

# just refresh the DQI layer on top of an already-built DB (fast, ~40s)
dbt build --select data_quality --target duckdb_local --vars '{enable_input_layer_testing: true}'

# then re-copy the artifacts next to this notebook:
cp <duckdb_local path from ~/.dbt/profiles.yml> tuva_demo.duckdb
cp target/run_results.json run_results.json   # only after a FULL build/test run
```

In [1]:
import json
import duckdb
import pandas as pd
from pathlib import Path

pd.set_option('display.max_rows', 100)
pd.set_option('display.width', 140)

NOTEBOOK_DIR = Path('.').resolve()
DB_PATH = NOTEBOOK_DIR / 'tuva_demo.duckdb'
RUN_RESULTS_PATH = NOTEBOOK_DIR / 'run_results.json'

con = duckdb.connect(str(DB_PATH), read_only=True)
print(f"Connected to {DB_PATH}")
print(f"Reading build results from {RUN_RESULTS_PATH}")

ModuleNotFoundError: No module named 'duckdb'

## 1. Build & test summary

In [2]:
run_results = json.loads(RUN_RESULTS_PATH.read_text())
print("Build generated at:", run_results['metadata']['generated_at'])

rows = []
for r in run_results['results']:
    rows.append({
        'unique_id': r['unique_id'],
        'resource_type': r['unique_id'].split('.')[0],
        'status': r['status'],
        'execution_time': round(r.get('execution_time', 0), 2),
        'message': (r.get('message') or '')[:200],
    })
results_df = pd.DataFrame(rows)

print("\nOverall status counts:")
display(results_df['status'].value_counts())

print("\nTest-specific status counts (uniqueness, not_null, relationships, dbt_utils/dbt_expectations, etc.):")
tests_df = results_df[results_df['resource_type'] == 'test']
display(tests_df['status'].value_counts())

Build generated at: 2026-09-17T23:07:04.015176Z

Overall status counts:


status
success    581
pass        48
warn         3
Name: count, dtype: int64


Test-specific status counts (uniqueness, not_null, relationships, dbt_utils/dbt_expectations, etc.):


status
pass    48
warn     3
Name: count, dtype: int64

In [3]:
# Any failures or errors, model or test -- this is the "did anything actually break" check
failures = results_df[results_df['status'].isin(['error', 'fail'])]
if failures.empty:
    print("No errors or failures. \u2713")
else:
    print(f"{len(failures)} error/fail results:")
    display(failures[['unique_id', 'status', 'message']])

No errors or failures. ✓


In [4]:
# Warnings are worth a look but aren't necessarily problems -- Tuva ships some soft DQ checks
# (e.g. "allowed_amount populated") as warn-severity by design.
warnings_df = results_df[results_df['status'] == 'warn']
print(f"{len(warnings_df)} warnings:")
display(warnings_df[['unique_id', 'message']])

3 warnings:


,unique_id,message
167,test.the_tuva_project.check_medical_allowed_am...,"Got 121674 results, configured to warn if != 0"
168,test.the_tuva_project.check_pharmacy_allowed_a...,"Got 9732 results, configured to warn if != 0"
572,test.the_tuva_project.the_tuva_project_expect_...,"Got 42651 results, configured to warn if != 0"


## 2. DQI input-layer data quality summary

`data_quality__data_quality_for_pbi` buckets every checked field into `valid` / `invalid` / `null` per table+field, from the raw claims data. This is Tuva's own DQI framework (`enable_input_layer_testing`), which we confirmed is safe on Databricks Free Edition (~1 extra table, no broken dependencies -- unlike `enable_legacy_data_quality`, which hard-depends on marts we have disabled).

In [5]:
dqi = con.execute("""
    select table_name, field_name, bucket_name, sum(frequency) as n
    from data_quality.data_quality_for_pbi
    group by table_name, field_name, bucket_name
""").fetchdf()

pivot = dqi.pivot_table(index=['table_name', 'field_name'], columns='bucket_name', values='n', fill_value=0, aggfunc='sum')
for col in ('valid', 'invalid', 'null'):
    if col not in pivot.columns:
        pivot[col] = 0
pivot['total'] = pivot[['valid', 'invalid', 'null']].sum(axis=1)
pivot['pct_valid'] = (pivot['valid'] / pivot['total'] * 100).round(1)
pivot = pivot.sort_values('pct_valid')

print(f"{len(pivot)} table/field combinations checked.")
print("\nWorst 20 by % valid (worth a look, not necessarily a problem -- some fields are legitimately sparse):")
display(pivot.head(20))

96 table/field combinations checked.

Worst 20 by % valid (worth a look, not necessarily a problem -- some fields are legitimately sparse):


bucket_name                             duplicate  invalid  multiple      null  valid     total  pct_valid
table_name     field_name                                                                                 
ELIGIBILITY    CITY                           0.0      0.0       0.0    2309.0    0.0    2309.0        0.0
               DUAL_STATUS_CODE               0.0    438.0       0.0    1871.0    0.0    2309.0        0.0
               EMAIL                          0.0      0.0       0.0    2309.0    0.0    2309.0        0.0
               ETHNICITY                      0.0      0.0       0.0    2309.0    0.0    2309.0        0.0
               LAST_NAME                      0.0      0.0       0.0    2309.0    0.0    2309.0        0.0
               MIDDLE_NAME                    0.0      0.0       0.0    2309.0    0.0    2309.0        0.0
               FIRST_NAME                     0.0      0.0       0.0    2309.0    0.0    2309.0        0.0
               NAME_SUFFIX                    0.0      0.0       0.0    2309.0    0.0    2309.0        0.0
               PHONE                          0.0      0.0       0.0    2309.0    0.0    2309.0        0.0
               ZIP_CODE                       0.0      0.0       0.0    2309.0    0.0    2309.0        0.0
MEDICAL_CLAIM  DIAGNOSIS_POA_3                0.0    189.0       0.0    8437.0    0.0    8626.0        0.0
               PAID_DATE                      0.0      0.0       0.0  167995.0    0.0  167995.0        0.0
               DIAGNOSIS_POA_1                0.0    209.0       0.0    8417.0    0.0    8626.0        0.0
               COINSURANCE_AMOUNT             0.0      0.0       0.0  167995.0    0.0  167995.0        0.0
               DEDUCTIBLE_AMOUNT              0.0      0.0       0.0  167995.0    0.0  167995.0        0.0
               COPAYMENT_AMOUNT               0.0      0.0       0.0  167995.0    0.0  167995.0        0.0
               DIAGNOSIS_POA_2                0.0    196.0       0.0    8430.0    0.0    8626.0        0.0
PHARMACY_CLAIM DEDUCTIBLE_AMOUNT              0.0      0.0       0.0   12019.0    0.0   12019.0        0.0
               DISPENSING_PROVIDER_NPI        0.0      0.0       0.0   12019.0    0.0   12019.0        0.0
               REFILLS                        0.0      0.0       0.0   12019.0    0.0   12019.0        0.0

In [6]:
# Fields that are majority-invalid (not just null) -- these are the ones actually worth investigating
pivot['pct_invalid'] = (pivot['invalid'] / pivot['total'] * 100).round(1)
flagged = pivot[pivot['pct_invalid'] > 5].sort_values('pct_invalid', ascending=False)
if flagged.empty:
    print("No table/field has >5% invalid (non-null, non-conforming) values. \u2713")
else:
    display(flagged)

bucket_name                           duplicate   invalid  multiple     null     valid     total  pct_valid  pct_invalid
table_name     field_name                                                                                               
MEDICAL_CLAIM  HCPCS_CODE                   0.0  129575.0       0.0  11655.0   26765.0  167995.0       15.9         77.1
PHARMACY_CLAIM DISPENSING_DATE              0.0    5466.0       0.0      0.0    6553.0   12019.0       54.5         45.5
               PAID_DATE                    0.0    5466.0       0.0      0.0    6553.0   12019.0       54.5         45.5
MEDICAL_CLAIM  CLAIM_START_DATE             0.0   30146.0       0.0      0.0  111954.0  142100.0       78.8         21.2
               CLAIM_END_DATE               0.0   30139.0       0.0      0.0  111961.0  142100.0       78.8         21.2
ELIGIBILITY    DUAL_STATUS_CODE             0.0     438.0       0.0   1871.0       0.0    2309.0        0.0         19.0
               RACE                         0.0     396.0       0.0      0.0    1913.0    2309.0       82.8         17.2
MEDICAL_CLAIM  CLAIM_LINE_END_DATE          0.0   28121.0       0.0  34521.0  105353.0  167995.0       62.7         16.7
               CLAIM_LINE_START_DATE        0.0   28121.0       0.0  34521.0  105353.0  167995.0       62.7         16.7
               ADMISSION_DATE               0.0      28.0     259.0     11.0     140.0     179.0       78.2         15.6
               DISCHARGE_DATE               0.0      28.0     259.0     11.0     140.0     179.0       78.2         15.6

## 3. Business-logic sanity checks

In [7]:
checks = []

def check(name, value, ok, note=''):
    checks.append({'check': name, 'value': value, 'ok': ok, 'note': note})
    print(f"{'\u2713' if ok else '\u2717'} {name}: {value}  {note}")

# --- core layer volumes ---
n_patients = con.execute("select count(*) from core.patient").fetchone()[0]
n_encounters = con.execute("select count(*) from core.encounter").fetchone()[0]
n_medical_claims = con.execute("select count(*) from core.medical_claim").fetchone()[0]
check('core.patient row count', n_patients, n_patients > 0)
check('core.encounter row count', n_encounters, n_encounters > 0)
check('core.medical_claim row count matches seed (167,995)', n_medical_claims, n_medical_claims == 167995)
check('encounters compress claims sensibly (encounters < claims)', f'{n_encounters} < {n_medical_claims}', n_encounters < n_medical_claims)

✓ core.patient row count: 1000  
✓ core.encounter row count: 54080  
✓ core.medical_claim row count matches seed (167,995): 167995  
✓ encounters compress claims sensibly (encounters < claims): 54080 < 167995  


In [8]:
# --- readmissions: all-cause 30-day readmission rate should be in a clinically plausible range ---
# readmission_summary is one row per qualifying index encounter (disqualified_encounter_flag=0 in
# the upstream augmented table); index_admission_flag marks index stays, readmit_30_flag marks
# whether that encounter's *next* encounter for the same person was <=30 days later.
readm = con.execute("""
    select
        count(*) as n_summary_rows,
        sum(index_admission_flag) as index_admissions,
        sum(case when index_admission_flag = 1 then readmit_30_flag else 0 end) as readmissions_30d,
        sum(case when index_admission_flag = 1 then readmit_30_flag else 0 end) * 100.0
            / nullif(sum(index_admission_flag), 0) as readmission_rate_pct
    from readmissions.readmission_summary
""").fetchdf()
display(readm)

rate = readm['readmission_rate_pct'].iloc[0]
# Published all-cause 30-day readmission benchmarks are roughly 10-20%; this is synthetic data so
# treat this as a sanity band, not a clinical claim.
check('readmission_summary row count', readm['n_summary_rows'].iloc[0], readm['n_summary_rows'].iloc[0] > 0)
check('readmission rate in plausible range (0-40%)', f'{rate:.1f}%' if rate is not None else 'n/a',
      rate is not None and 0 <= rate <= 40)

,n_summary_rows,index_admissions,readmissions_30d,readmission_rate_pct
0,201,190.0,13.0,6.842105


✓ readmission_summary row count: 201  
✓ readmission rate in plausible range (0-40%): 6.8%  


In [9]:
# --- ccsr: every mapped condition should trace back to a real CCSR category ---
ccsr = con.execute("""
    select count(*) as n_rows,
           count(distinct ccsr_category) as n_categories,
           sum(case when ccsr_category is null then 1 else 0 end) as null_categories
    from ccsr.long_condition_category
""").fetchdf()
display(ccsr)
check('ccsr.long_condition_category row count matches Databricks (229,634)', ccsr['n_rows'].iloc[0],
      ccsr['n_rows'].iloc[0] == 229634)
check('no null ccsr categories', ccsr['null_categories'].iloc[0], ccsr['null_categories'].iloc[0] == 0)

,n_rows,n_categories,null_categories
0,229634,411,0.0


✓ ccsr.long_condition_category row count matches Databricks (229,634): 229634  
✓ no null ccsr categories: 0.0  


In [10]:
# --- provider attribution: what share of patients actually got attributed? ---
attributed = con.execute("select count(distinct person_id) from tuva_provider_attribution.assigned_beneficiaries_current").fetchone()[0]
coverage_pct = attributed / n_patients * 100
check('patients attributed', attributed, attributed > 0)
check('attribution coverage', f'{coverage_pct:.1f}%', coverage_pct > 50,
      note='(low coverage would suggest a join/logic problem, not necessarily an error)')

✓ patients attributed: 974  
✓ attribution coverage: 97.4%  (low coverage would suggest a join/logic problem, not necessarily an error)


In [11]:
# --- ahrq pqi: pqi_summary is qualifying-encounter detail; pqi_rate has the actual computed rates ---
pqi_summary = con.execute("select count(*) as n from ahrq_measures.pqi_summary").fetchone()[0]
check('pqi_summary row count', pqi_summary, pqi_summary > 0)

pqi_rate = con.execute("""
    select pqi_number, denom_count, num_count, rate_per_100_thousand
    from ahrq_measures.pqi_rate
    order by pqi_number
""").fetchdf()
display(pqi_rate)
check('pqi_rate row count', len(pqi_rate), len(pqi_rate) > 0)
check('all pqi rates non-negative', (pqi_rate['rate_per_100_thousand'] >= 0).all(),
      (pqi_rate['rate_per_100_thousand'] >= 0).all())
check('no zero denominators', (pqi_rate['denom_count'] > 0).all(), (pqi_rate['denom_count'] > 0).all())

✓ pqi_summary row count: 32  


,pqi_number,denom_count,num_count,rate_per_100_thousand
0,01,974,0,0.000000
1,01,999,0,0.000000
2,01,942,0,0.000000
3,03,942,0,0.000000
4,03,999,0,0.000000
5,03,974,0,0.000000
6,05,983,3,305.188199
7,05,959,2,208.550574
8,05,924,1,108.225108
9,07,999,1,100.100100


✓ pqi_rate row count: 30  
✓ all pqi rates non-negative: True  
✓ no zero denominators: True  


## 4. Databricks cross-check

Static reference numbers from the last verified Databricks run (re-confirmed live on 2026-09-18). Re-run the queries in `notebooks/databricks_check.py` (or ask for a live pull) if you want a fresh comparison instead of this snapshot.

In [12]:
databricks_reference = {
    'core.encounter': 54080,
    'readmissions.readmission_summary': 198,
    'ccsr.long_condition_category': 229634,
    'tuva_provider_attribution.assigned_beneficiaries_current': 974,
    'ahrq_measures.pqi_summary': 32,
}
local_counts = {
    'core.encounter': n_encounters,
    'readmissions.readmission_summary': int(readm['n_summary_rows'].iloc[0]),
    'ccsr.long_condition_category': int(ccsr['n_rows'].iloc[0]),
    'tuva_provider_attribution.assigned_beneficiaries_current': attributed,
    'ahrq_measures.pqi_summary': pqi_summary,
}
compare = pd.DataFrame({
    'databricks': databricks_reference,
    'duckdb_local': local_counts,
})
compare['match'] = compare['databricks'] == compare['duckdb_local']
display(compare)

,databricks,duckdb_local,match
core.encounter,54080,54080,True
readmissions.readmission_summary,198,201,False
ccsr.long_condition_category,229634,229634,True
tuva_provider_attribution.assigned_beneficiaries_current,974,974,True
ahrq_measures.pqi_summary,32,32,True


## Summary verdict

In [13]:
checks_df = pd.DataFrame(checks)
n_fail = (~checks_df['ok']).sum()
print(f"{len(checks_df) - n_fail} / {len(checks_df)} business-logic checks passed.")
print(f"{len(failures)} build/test errors or failures.")
print(f"{int((~compare['match']).sum())} / {len(compare)} Databricks cross-checks mismatched.")
if n_fail == 0 and failures.empty and compare['match'].all():
    print("\n\u2705 All checks passed.")
else:
    print("\n\u26a0\ufe0f Something above needs a look.")

14 / 14 business-logic checks passed.
0 build/test errors or failures.
1 / 5 Databricks cross-checks mismatched.

⚠️ Something above needs a look.
